# Stage 2: Signal Processing & Feature Engineering
# Here, we apply the Hodrick-Prescott (HP) filter to separate long-term economic trends from short-term market static, ensuring the model trains on true systemic imbalances.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Load cleaned data from Notebook 1
df = pd.read_csv('data/01_cleaned_raw.csv')

# 2. Feature Engineering: Credit Velocity
df['credit_gdp'] = df['tloans'] / df['gdp']
df['credit_gdp_diff2'] = df.groupby('country')['credit_gdp'].diff(2)

# 3. Digital Filtering (HP Filter)
def apply_hp_filter(series, lamb=100):
    if series.dropna().empty:
        return series
    cycle, _ = sm.tsa.filters.hpfilter(series.dropna(), lamb=lamb)
    return cycle.reindex(series.index)

print("Applying HP Filters...")
df['credit_gdp_cycle'] = df.groupby('country')['credit_gdp'].transform(apply_hp_filter)
df['yield_curve_slope'] = df['ltrate'] - df['stir']
df['yield_curve_cycle'] = df.groupby('country')['yield_curve_slope'].transform(apply_hp_filter)

# 4. Define final features and impute missing values
features = ['yield_curve_slope', 'credit_gdp', 'credit_gdp_diff2', 'credit_gdp_cycle', 'yield_curve_cycle', 'cpi', 'unemp', 'debtgdp']
for col in features:
    df[col] = df[col].fillna(df[col].median())

# 5. Save the final processed signals
df.to_csv('data/processed_signals.csv', index=False)
print("Preprocessing complete. Signals saved.")
df[features].describe()

Applying HP Filters...
Preprocessing complete. Signals saved.


,yield_curve_slope,credit_gdp,credit_gdp_diff2,credit_gdp_cycle,yield_curve_cycle,cpi,unemp,debtgdp
count,2668.000000,2668.000000,2668.000000,2668.000000,2668.000000,2.668000e+03,2668.000000,2668.000000
mean,0.787826,0.581455,0.013440,-0.000107,0.000973,4.188763e+01,5.246000,0.536417
std,1.890273,0.346635,0.066533,0.044235,1.022478,5.688478e+01,3.509940,0.380076
min,-10.924842,0.004682,-0.527698,-0.352958,-8.028513,1.150360e-11,0.035923,0.019074
25%,-0.134226,0.326851,-0.009947,-0.018526,-0.379288,2.526395e+00,3.140000,0.269825
50%,0.734267,0.528971,0.013875,-0.001282,0.014664,1.050679e+01,4.600000,0.447780
75%,1.614471,0.767318,0.040555,0.015918,0.402479,7.599186e+01,6.272639,0.682662
max,17.821839,2.035726,0.402512,0.380052,7.169720,2.268252e+02,26.093600,2.697976
